# 🔥 guato_ba — Modelos de Machine Learning
### Classificação por pixel com 11 atributos espectrais
<hr style="border:1px solid #e69627;">

<div style="text-align: justify; margin-right: 2.5%;">

Ao lado das redes profundas (que olham o **patch inteiro**, com contexto espacial), o estudo compara **4 classificadores clássicos** que decidem **pixel a pixel**: `rf` (Random Forest), `xgboost`, `mlp` e `svm`. Cada pixel vira um **vetor de 11 atributos** e o modelo diz: queimou ou não.

</div>

# <span style="color:#cf4c39">🧮 1. Os 11 atributos</span>
<hr style="border:1px solid #e69627;">

<div style="text-align: justify; margin-right: 2.5%;">

Para cada pixel montamos:

| # | atributos | o que é |
|---|---|---|
| 1–4 | reflectância de **T1** | B04, B8A, B11, B12 na data 1 |
| 5–8 | reflectância de **T2** | B04, B8A, B11, B12 na data 2 |
| 9 | **dNBR** | queda do *Normalized Burn Ratio* (T1−T2) |
| 10 | **dNBRSWIR** | variante SWIR do NBR (T1−T2) |
| 11 | **dMIRBI** | *Mid-InfraRed Burn Index* (T1−T2) |

Os três últimos são **índices de mudança**: medem o quanto a assinatura espectral de queimada apareceu entre as datas. As fórmulas estão em `geo_data_science.indices()`; abaixo reproduzimos tudo com numpy puro a partir de um patch real do Dataset A.

</div>

In [1]:
import numpy as np
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT/"data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DATA = ROOT/"data"

T1 = np.load(DATA/"train_v123_256_T1.npy", mmap_mode="r")
T2 = np.load(DATA/"train_v123_256_T2.npy", mmap_mode="r")

def refl(dn):        # DN -> reflectância
    return dn.astype("float32")/10000.0

def indices(n, s1, s2):
    """n=B8A, s1=B11, s2=B12 (reflectâncias). Retorna NBR, NBRSWIR, MIRBI."""
    nbr = (n - s2) / (n + s2 + 1e-6)
    nsw = (s1 - s2 - 0.02) / (s1 + s2 + 0.1)
    mir = 10.0*s2 - 9.8*s1 + 2.0
    return nbr, nsw, mir

def features_11(p1, p2):
    """p1,p2: (4,H,W) [B04,B8A,B11,B12] -> (11,H,W)."""
    r1 = refl(p1); r2 = refl(p2)
    nbr1, nsw1, mir1 = indices(r1[1], r1[2], r1[3])
    nbr2, nsw2, mir2 = indices(r2[1], r2[2], r2[3])
    return np.stack([*r1, *r2, nbr1-nbr2, nsw1-nsw2, mir1-mir2], axis=0)

F = features_11(np.asarray(T1[0]), np.asarray(T2[0]))
nomes = ["t1_B04","t1_B8A","t1_B11","t1_B12","t2_B04","t2_B8A","t2_B11","t2_B12","dNBR","dNBRSWIR","dMIRBI"]
print("mapa de atributos:", F.shape, "(11 atributos, 256x256)")
print("\nvalor médio de cada atributo neste patch:")
for nm, v in zip(nomes, F.reshape(11, -1).mean(1)):
    print(f"  {nm:10s} {v:8.3f}")

mapa de atributos: (11, 256, 256) (11 atributos, 256x256)

valor médio de cada atributo neste patch:
  t1_B04        0.084
  t1_B8A        0.259
  t1_B11        0.168
  t1_B12        0.112
  t2_B04        0.092
  t2_B8A        0.244
  t2_B11        0.182
  t2_B12        0.122
  dNBR          0.041
  dNBRSWIR     -0.001
  dMIRBI        0.031


<div style="text-align: justify; margin-right: 2.5%;">

Para treinar/avaliar, o pipeline achata isso em uma tabela `(n_pixels, 11)` considerando só pixels válidos (`Y != 255`) — é o que faz `extrair_features_pixel()` em `geo_data_science.py`, com **balanceamento** de classes no treino e **proporção natural** na avaliação.

</div>

# <span style="color:#cf4c39">🌳 2. Os 4 modelos e seus melhores hiperparâmetros</span>
<hr style="border:1px solid #e69627;">

<div style="text-align: justify; margin-right: 2.5%;">

As fábricas estão em `redes.MODELOS_ML`. Os melhores hiperparâmetros (buscados com Optuna, maximizando o IoU de validação) ficam versionados em `machine_learning/<modelo>/hpo_best.json`.

</div>

In [2]:
import json, sys
sys.path.insert(0, str(ROOT))
import redes

print("modelos ML disponíveis:", list(redes.MODELOS_ML))
print()
for nome in redes.MODELOS_ML:
    hp_path = ROOT/f"machine_learning/{nome}/hpo_best.json"
    if hp_path.exists():
        d = json.load(open(hp_path))
        print(f"{nome:8s} IoU_val={d.get('best_iou_val', float('nan')):.4f}  ->  {d['best_params']}")
    else:
        print(f"{nome:8s} (sem hpo_best.json)")

modelos ML disponíveis: ['rf', 'xgboost', 'mlp', 'svm']

rf       IoU_val=0.9919  ->  {'n_estimators': 270, 'max_depth': 21, 'min_samples_leaf': 1, 'max_features': 'sqrt'}
xgboost  IoU_val=0.9917  ->  {'n_estimators': 400, 'max_depth': 7, 'learning_rate': 0.02023853396323038, 'subsample': 0.6529402702220294, 'colsample_bytree': 0.6801503150700818, 'min_child_weight': 3}
mlp      IoU_val=0.9917  ->  {'hidden': '256,128,64', 'alpha': 0.00011151498971978547, 'lr': 0.003623266987127362}
svm      IoU_val=0.9898  ->  {'C': 64.733437270269, 'gamma': 'scale'}


<div style="text-align: justify; margin-right: 2.5%;">

💡 Repare que o **RF** atinge IoU de validação altíssimo em 2024 — mas o teste honesto é 2025 (notebook 04), onde os modelos por pixel **superestimam** a área queimada (comissão alta), justamente por não enxergarem o **contexto espacial** que as U-Nets usam.

</div>

---
Continue no **`04_metricas_e_avaliacao`** para comparar DL × ML de forma justa.